In [1]:
from pathlib import Path
import json
import re
import unicodedata

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

extracted_path = (
    PROJECT_ROOT / "data" / "processed" / "extracted_pages.json"
)

with extracted_path.open("r", encoding="utf-8") as file:
    raw_pages = json.load(file)

print("Page records loaded:", len(raw_pages))
print("First document:", raw_pages[0]["source"])
print("First page:", raw_pages[0]["page_number"])

Page records loaded: 4
First document: data_pipeline_runbook.pdf
First page: 1


In [2]:
sample_text = raw_pages[0]["text"]

print("Normal display:\n")
print(sample_text)

print("\nRepresentation of the text:\n")
print(repr(sample_text))

Normal display:

DATA PIPELINE RUNBOOK
Pipeline: CUSTOMER_DAILY_LOAD
Schedule: Daily at 02:00 UTC.
Upstream dependency: CUSTOMER_RAW_INGEST.
Normal operation:
1. Confirm the upstream ingestion completed successfully.
2. Confirm the expected business-date partition exists.
3. Start the downstream load after dependencies are healthy.
Owner: Data Operations.

Representation of the text:

'DATA PIPELINE RUNBOOK\nPipeline: CUSTOMER_DAILY_LOAD\nSchedule: Daily at 02:00 UTC.\nUpstream dependency: CUSTOMER_RAW_INGEST.\nNormal operation:\n1. Confirm the upstream ingestion completed successfully.\n2. Confirm the expected business-date partition exists.\n3. Start the downstream load after dependencies are healthy.\nOwner: Data Operations.'


In [3]:
def clean_pdf_text(text: str) -> str:
    """Normalize common whitespace and text-formatting artifacts."""

    if not isinstance(text, str):
        raise TypeError("Expected PDF text to be a string.")

    # Normalize compatible Unicode character representations.
    text = unicodedata.normalize("NFC", text)

    # Standardize different newline formats.
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Replace non-breaking spaces and remove soft hyphens.
    text = text.replace("\u00a0", " ")
    text = text.replace("\u00ad", "")

    # Normalize spaces and tabs without removing line boundaries.
    lines = [
        re.sub(r"[ \t]+", " ", line).strip()
        for line in text.split("\n")
    ]

    text = "\n".join(lines)

    # Collapse excessive blank lines, preserving paragraph breaks.
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [4]:
cleaned_pages = []

for record in raw_pages:
    cleaned_record = record.copy()

    cleaned_record["text"] = clean_pdf_text(
        record["text"]
    )

    cleaned_pages.append(cleaned_record)

print("Original records:", len(raw_pages))
print("Cleaned records:", len(cleaned_pages))

Original records: 4
Cleaned records: 4


In [5]:
sample_index = 0

print("BEFORE CLEANING:\n")
print(repr(raw_pages[sample_index]["text"]))

print("\nAFTER CLEANING:\n")
print(repr(cleaned_pages[sample_index]["text"]))

BEFORE CLEANING:

'DATA PIPELINE RUNBOOK\nPipeline: CUSTOMER_DAILY_LOAD\nSchedule: Daily at 02:00 UTC.\nUpstream dependency: CUSTOMER_RAW_INGEST.\nNormal operation:\n1. Confirm the upstream ingestion completed successfully.\n2. Confirm the expected business-date partition exists.\n3. Start the downstream load after dependencies are healthy.\nOwner: Data Operations.'

AFTER CLEANING:

'DATA PIPELINE RUNBOOK\nPipeline: CUSTOMER_DAILY_LOAD\nSchedule: Daily at 02:00 UTC.\nUpstream dependency: CUSTOMER_RAW_INGEST.\nNormal operation:\n1. Confirm the upstream ingestion completed successfully.\n2. Confirm the expected business-date partition exists.\n3. Start the downstream load after dependencies are healthy.\nOwner: Data Operations.'


In [6]:
assert len(raw_pages) == len(cleaned_pages)

assert all(
    original["source"] == cleaned["source"]
    and original["page_number"] == cleaned["page_number"]
    for original, cleaned in zip(raw_pages, cleaned_pages)
)

assert all(
    isinstance(record["text"], str)
    for record in cleaned_pages
)

empty_pages = [
    (record["source"], record["page_number"])
    for record in cleaned_pages
    if not record["text"]
]

print("Record count preserved.")
print("Source and page metadata preserved.")
print("Empty cleaned pages:", empty_pages)

Record count preserved.
Source and page metadata preserved.
Empty cleaned pages: []


In [7]:
cleaned_path = (
    PROJECT_ROOT / "data" / "processed" / "cleaned_pages.json"
)

with cleaned_path.open("w", encoding="utf-8") as file:
    json.dump(
        cleaned_pages,
        file,
        ensure_ascii=False,
        indent=2
    )

print("Saved cleaned pages to:", cleaned_path)

Saved cleaned pages to: c:\Users\USER\Desktop\Gen_AI\DocuTrace_Rag\data\processed\cleaned_pages.json
